##**Step 1: Problem Understanding and Framing**

This section defines the business problem, data science problem, machine learning task, target variable, success metrics, and business KPIs for the capstone project. The selected domain is Finance: Detect fraudulent transactions, with a specific focus on detecting fraudulent bank account applications.

**a. Problem Definition and Objectives**


Fraudulent bank account applications create significant financial, operational, regulatory, and reputational risks for financial institutions. Once opened, fraudulent accounts may be used for identity fraud, mule activity, unauthorized transactions, money laundering, and other illicit financial activities. These risks are especially relevant in digital account opening, where applications can be submitted quickly and fraud patterns may evolve faster than traditional rule-based controls can detect.


In many financial institutions, fraud detection relies on rule-based screening, manual review, and analyst judgment. These controls remain important, but they may not be sufficient to detect complex, hidden, or evolving fraud patterns across large volumes of applications. This creates an opportunity to use machine learning as a decision-support tool that can assign fraud risk scores to applications and help fraud analysts prioritize the highest-risk cases for review.


This capstone project will use the Bank Account Fraud Dataset Suite - NeurIPS 2022. For the main analysis, Base.csv will be used as the primary dataset because it serves as the baseline dataset in the BAF suite. Variant II.csv will be used as a secondary robustness and fairness stress-test dataset. The two datasets will not be merged because each BAF variant is designed to represent a different controlled data setting. Keeping them separate allows the project to compare fraud prevalence, feature distributions, model performance, and fairness behavior across dataset variants.


From a data science perspective, this project is a supervised binary classification problem. The target variable is fraud_bool, where 1 indicates a fraudulent application and 0 indicates a legitimate application. The model will learn from application, customer, behavioral, device, session, credit-risk, and time-related features to estimate the probability that an account application is fraudulent.


The main objective is to develop an accurate, explainable, and responsible fraud detection model. The model will not be positioned as an automated rejection system. Instead, it will serve as a decision-support tool that helps fraud analysts prioritize suspicious applications for manual review. This is important because fraud detection is a high-stakes financial use case. A false negative may allow a fraudulent application to pass undetected, while a false positive may inconvenience a legitimate customer.


The project will compare multiple classification models, beginning with interpretable baseline models and progressing to more advanced models. Logistic Regression will be used as an initial baseline because it is explainable and easy to interpret. Decision Tree, Random Forest, XGBoost, or similar models may be used to test whether more complex algorithms can improve fraud detection performance. The final model will be selected based on technical performance, business usefulness, explainability, fairness, and robustness across the Base and Variant II datasets.

**b. Business and Data Science Problem Framing**


The business problem is that fraudulent bank account applications can expose financial institutions to losses, operational burden, compliance concerns, and reputational damage. Fraud teams usually have limited capacity and cannot manually review every application in detail. Therefore, the model should help prioritize applications that are more likely to be fraudulent.


The data science problem is to predict whether an account application is fraudulent or legitimate using available features in the dataset. Since the output has two possible classes, fraud or non-fraud, the project is a binary classification task. The model output will be a fraud probability or fraud risk score. This score can then be used to rank applications from highest risk to lowest risk.


The analytical objective is not only to build a high-performing model, but also to understand which factors contribute to fraud predictions, evaluate model performance under class imbalance, and assess whether the model behaves responsibly across relevant customer groups. This prepares the project for later capstone steps, including data understanding, preprocessing, feature engineering, model comparison, explainability, and bias auditing.

**c. Success Metrics and Business KPIs**


The project will use both technical metrics and business metrics.


Because fraud cases are typically rare, accuracy alone will not be used as the primary metric. A model can achieve high accuracy by predicting most applications as legitimate while still missing many actual fraud cases. Therefore, the primary technical metric will be PR-AUC / Average Precision, because it focuses more on performance for the minority fraud class.


Supporting technical metrics will include ROC-AUC, recall, precision, F1-score, confusion matrix, false positive rate, false negative rate, and recall at top-K% highest-risk applications. Recall is important because the bank wants to detect as many fraud cases as possible. Precision is important because too many false positives can overload fraud analysts and inconvenience legitimate customers. F1-score balances precision and recall. The confusion matrix will show true positives, false positives, true negatives, and false negatives.


The project will also use recall at top-K% highest-risk applications. This is highly relevant to banking operations because fraud teams usually have limited review capacity. For example, if analysts can only review the top 10% highest-risk applications, this metric will show how many actual fraud cases are captured within that review queue.


The business KPIs will include fraud detection uplift, estimated fraud loss avoided, manual review efficiency, investigation workload, false positive impact, estimated net benefit, and responsible AI readiness. These KPIs will help translate model results into operational and executive-level value.


Estimated Net Benefit = Estimated Fraud Loss Avoided − Manual Review Cost − False Positive Cost

**d. Reproducibility Setup**

Before loading and analyzing the dataset, the notebook will document the Python environment, random seed, and required libraries. This supports reproducibility because model results may vary due to random sampling, train-test splits, model initialization, and package versions.


A fixed random seed will be used throughout the project to make the workflow more consistent and easier to validate. The project will also import libraries for data handling, visualization, machine learning, model evaluation, explainability, class imbalance handling, and fairness analysis.



In [ ]:
import sys
import platform
import random
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Python version:", sys.version)
print("Platform:", platform.platform())
print("Random seed:", SEED)

Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
Random seed: 42


In [ ]:
# Install required libraries

%pip install -q pandas numpy scikit-learn matplotlib seaborn shap xgboost fairlearn imbalanced-learn


import pandas as pd
import numpy as np

import sklearn
import matplotlib
import shap

from pathlib import Path

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("matplotlib:", matplotlib.__version__)
print("shap:", shap.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.5/135.5 kB 1.3 MB/s eta 0:00:00
pandas: 2.2.2
numpy: 2.0.2
scikit-learn: 1.6.1
matplotlib: 3.10.0
shap: 0.52.0


**e. Technical Evaluation Functions**


The following functions will be used later in model evaluation. The actual metric values will be computed in Step 4 after the models generate fraud probabilities.


The evaluate_classification_model function computes the key classification metrics for fraud detection. It accepts the actual labels, predicted fraud probabilities, and a classification threshold. It returns accuracy, PR-AUC, ROC-AUC, precision, recall, F1-score, confusion matrix values, false positive rate, and false negative rate.


In [ ]:
# Technical evaluation function

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

def evaluate_classification_model(y_true, y_score, threshold=0.50):
    """
    Evaluates binary fraud classification performance.

    Parameters:
    y_true    : Actual labels, where 1 = fraud and 0 = legitimate
    y_score   : Predicted fraud probability or fraud risk score
    threshold : Cut-off used to convert probability into predicted class

    Returns:
    Dictionary of technical model metrics.
    """

    y_pred = (y_score >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    metrics = {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "pr_auc_average_precision": average_precision_score(y_true, y_score),
        "roc_auc": roc_auc_score(y_true, y_score),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
        "true_positives": tp,
        "false_positives": fp,
        "true_negatives": tn,
        "false_negatives": fn,
        "false_positive_rate": fp / (fp + tn) if (fp + tn) > 0 else 0,
        "false_negative_rate": fn / (fn + tp) if (fn + tp) > 0 else 0
    }

    return metrics

print("Classification evaluation function initialized.")

Classification evaluation function initialized.


**f. Initial Fraud Review Policy and Top-K Metric**

To connect the model output to an actual fraud operations workflow, the project will evaluate an initial review policy where analysts review the top 10% highest-risk applications. This is more realistic than assuming every application can be manually investigated.


The recall at top-K% metric measures how many actual fraud cases are captured if analysts review only the highest-risk applications. This metric is useful because it reflects investigation capacity. If the model captures a high percentage of fraud cases within the top 10% highest-risk applications, it means the model can help fraud analysts work more efficiently.

In [ ]:
# Initial fraud review policy and top-K metric

TOP_REVIEW_RATE = 0.10

print(f"Initial review policy: flag the top {int(TOP_REVIEW_RATE * 100)}% highest-risk applications for manual review.")

def recall_at_top_k(y_true, y_score, review_rate=0.10):
    """
    Measures how many fraud cases are captured if analysts review only
    the top K% highest-risk applications.

    Parameters:
    y_true      : Actual labels, where 1 = fraud and 0 = legitimate
    y_score     : Predicted fraud probability or risk score
    review_rate : Percentage of applications selected for manual review

    Returns:
    Dictionary of top-K review metrics.
    """

    results = pd.DataFrame({
        "y_true": y_true,
        "y_score": y_score
    }).sort_values("y_score", ascending=False)

    n_review = int(np.ceil(len(results) * review_rate))
    reviewed = results.head(n_review)

    total_fraud = results["y_true"].sum()
    fraud_captured = reviewed["y_true"].sum()

    recall_top_k = fraud_captured / total_fraud if total_fraud > 0 else 0
    precision_top_k = fraud_captured / n_review if n_review > 0 else 0
    detection_uplift = recall_top_k / review_rate if review_rate > 0 else 0

    return {
        "review_rate": review_rate,
        "applications_reviewed": n_review,
        "fraud_captured": int(fraud_captured),
        "total_fraud": int(total_fraud),
        "recall_at_top_k": recall_top_k,
        "precision_at_top_k": precision_top_k,
        "detection_uplift_vs_random": detection_uplift
    }

print("Top-K fraud capture function initialized.")

Initial review policy: flag the top 10% highest-risk applications for manual review.
Top-K fraud capture function initialized.


**g. Business KPI Estimation Function**


The project will also estimate business KPIs to show practical value. These estimates will translate model results into operational terms such as estimated fraud loss avoided, manual review cost, false positive impact, and estimated net benefit.


The cost assumptions in this section are placeholders for capstone demonstration purposes. They are not actual bank values. They can be adjusted later depending on the assumed business scenario. The purpose is to show how model performance can be converted into business impact.

In [ ]:
# Business KPI estimation function

def business_kpi_estimate(
    y_true,
    y_score,
    review_rate=0.10,
    estimated_loss_per_fraud=10000,
    manual_review_cost=100,
    false_positive_cost=250
):
    """
    Estimates business KPIs for fraud review prioritization.

    Parameters:
    y_true                   : Actual labels, where 1 = fraud and 0 = legitimate
    y_score                  : Predicted fraud probability or risk score
    review_rate              : Percentage of applications selected for manual review
    estimated_loss_per_fraud : Assumed loss avoided per detected fraud case
    manual_review_cost       : Assumed cost per manual review
    false_positive_cost      : Assumed cost of incorrectly flagging a legitimate application

    Returns:
    Dictionary of business KPI estimates.
    """

    results = pd.DataFrame({
        "y_true": y_true,
        "y_score": y_score
    }).sort_values("y_score", ascending=False)

    n_review = int(np.ceil(len(results) * review_rate))
    reviewed = results.head(n_review)

    fraud_captured = reviewed["y_true"].sum()
    legitimate_flagged = n_review - fraud_captured

    fraud_loss_avoided = fraud_captured * estimated_loss_per_fraud
    total_review_cost = n_review * manual_review_cost
    total_false_positive_cost = legitimate_flagged * false_positive_cost
    estimated_net_benefit = fraud_loss_avoided - total_review_cost - total_false_positive_cost

    return {
        "review_rate": review_rate,
        "applications_reviewed": n_review,
        "fraud_captured": int(fraud_captured),
        "legitimate_applications_flagged": int(legitimate_flagged),
        "estimated_fraud_loss_avoided": fraud_loss_avoided,
        "manual_review_cost": total_review_cost,
        "false_positive_cost": total_false_positive_cost,
        "estimated_net_benefit": estimated_net_benefit
    }

print("Business KPI estimation function initialized.")

Business KPI estimation function initialized.
